In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
p = pathlib.Path.cwd()
for q in (p, *p.parents):
    s = q / "src" / "ftbp"   # <- change "ftbp" if you rename the package
    if s.exists():
        sys.path.insert(0, str(s.parent))  # add .../src
        break
else:
    raise RuntimeError("src/ftbp not found")

In [ ]:
import numpy as np
import pandas as pd
import itertools
from math import comb
from scipy.optimize import brentq
from scipy.stats import norm, cauchy, uniform
from ftbp.estimators import estimate_theta, estimate_sigma
from ftbp.wald import *

In [ ]:
# --- Main experiment: two-sample breakdown point analysis (template) ---
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from tqdm import tqdm

# Example parameters (adapt as needed for your two-sample setup)
loss_type = 'huber'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.4811
ns = [50, 100, 200]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm]

# Example: two-sample effect sizes (adapt as needed)
effect_sizes = [-2, -1.5, -1, -.5, -.25, -.1, 0, .1, .25, .5, 1, 1.5, 2]

all_results = []
for effect in effect_sizes:
    for n in ns:
        for dist in dists:
            for seed in tqdm(seeds):
                np.random.seed(seed)
                phi0 = 0
                while phi0 == 0:
                    # --- Generate two samples ---
                    x1 = dist.rvs(size=n)
                    x2 = dist.rvs(size=n) + effect  # shift for effect size
                    phi0 = wald_test_two_sample(x1, x2, delta=delta, loss_type=loss_type)
                    if phi0 == 1:
                        bp_ub, _ = bound_power_upper_two_sample(x1, x2, delta=delta, loss_type=loss_type)
                        bp_lb, _ = bound_power_lower_two_sample(x1, x2, delta=delta, loss_type=loss_type)
                all_results.append({
                    'n': n,
                    'dist': getattr(dist, 'name', dist.__class__.__name__),
                    'seed': seed,
                    'bp_lb': bp_lb,
                    'bp_ub': bp_ub,
                    'effect': effect
                })

df = pd.DataFrame(all_results)

In [ ]:
# --- Aggregate and plot ---
df_avg = (
    df.groupby(['n', 'effect'], as_index=False)
    .agg({'bp_ub': 'mean', 'bp_lb': 'mean'})
)

df_avg = df_avg.rename(columns={
    'effect': 'Effect Size',
    'bp_ub': 'Upper Bound of BP',
    'bp_lb': 'Lower Bound of BP'
})

sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})  # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk") 
palette = sns.color_palette("husl", len(ns))
marker_map = {50: 'o', 100: 's', 200: '^'}

plt.figure(figsize=(10, 6))
ax = sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Lower Bound of BP',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend='full',
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Upper Bound of BP',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend=False,
    alpha=0.8,
)

i = -1
for n_val in [50, 100, 200]:
    i += 1
    sub = df_avg[df_avg['n'] == n_val]
    ax.fill_between(sub['Effect Size'],
                    sub['Lower Bound of BP'],
                    sub['Upper Bound of BP'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'Effect Size $\theta$')
plt.ylabel(r'$m$')
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False)
plt.tight_layout()
plt.savefig(f'two_sample_bp_{loss_type}.pdf', bbox_inches='tight')

In [ ]:
marker_map = {50: 'o', 100: 's', 200: '^'}

# make y-axis to bound/n, fraction
df_avg['Lower Bound of BP ratio'] = df_avg['Lower Bound of BP'] / df_avg['n']
df_avg['Upper Bound of BP ratio'] = df_avg['Upper Bound of BP'] / df_avg['n']

# 3. Single‐plot
plt.figure(figsize=(10, 6))
ax = sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Lower Bound of BP ratio',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend='full',
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Upper Bound of BP ratio',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend=False,
    alpha=0.8,
)

i = -1
for n_val in [50, 100, 200]:
    i += 1
    sub = df_avg[df_avg['n'] == n_val]
    ax.fill_between(sub['Effect Size'],
                    sub['Lower Bound of BP ratio'],
                    sub['Upper Bound of BP ratio'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'Effect Size $\theta$')
plt.ylabel('Breakdown Point of Rejection')
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False)
plt.tight_layout()
# plt.show()
plt.savefig(f'two_sample_bp_{loss_type}_ratio.pdf', bbox_inches='tight')

In [ ]:
# --- Main experiment: two-sample breakdown point analysis (template) ---
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from tqdm import tqdm

# Example parameters (adapt as needed for your two-sample setup)
loss_type = 'logcosh'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.4811
ns = [50, 100, 200]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm]

# Example: two-sample effect sizes (adapt as needed)
effect_sizes = [-2, -1.5, -1, -.5, -.25, -.1, 0, .1, .25, .5, 1, 1.5, 2]

all_results = []
for effect in effect_sizes:
    for n in ns:
        for dist in dists:
            for seed in tqdm(seeds):
                np.random.seed(seed)
                phi0 = 0
                while phi0 == 0:
                    # --- Generate two samples ---
                    x1 = dist.rvs(size=n)
                    x2 = dist.rvs(size=n) + effect  # shift for effect size
                    phi0 = wald_test_two_sample(x1, x2, delta=delta, loss_type=loss_type)
                    if phi0 == 1:
                        bp_ub, _ = bound_power_upper_two_sample(x1, x2, delta=delta, loss_type=loss_type)
                        bp_lb, _ = bound_power_lower_two_sample(x1, x2, delta=delta, loss_type=loss_type)
                all_results.append({
                    'n': n,
                    'dist': getattr(dist, 'name', dist.__class__.__name__),
                    'seed': seed,
                    'bp_lb': bp_lb,
                    'bp_ub': bp_ub,
                    'effect': effect
                })

df = pd.DataFrame(all_results)

In [ ]:
# --- Aggregate and plot ---
df_avg = (
    df.groupby(['n', 'effect'], as_index=False)
    .agg({'bp_ub': 'mean', 'bp_lb': 'mean'})
)

df_avg = df_avg.rename(columns={
    'effect': 'Effect Size',
    'bp_ub': 'Upper Bound of BP',
    'bp_lb': 'Lower Bound of BP'
})

sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})  # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk") 
palette = sns.color_palette("husl", len(ns))
marker_map = {50: 'o', 100: 's', 200: '^'}

plt.figure(figsize=(10, 6))
ax = sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Lower Bound of BP',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend='full',
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Upper Bound of BP',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend=False,
    alpha=0.8,
)

i = -1
for n_val in [50, 100, 200]:
    i += 1
    sub = df_avg[df_avg['n'] == n_val]
    ax.fill_between(sub['Effect Size'],
                    sub['Lower Bound of BP'],
                    sub['Upper Bound of BP'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'Effect Size $\theta$')
plt.ylabel(r'$m$')
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False)
plt.tight_layout()
plt.savefig(f'two_sample_bp_{loss_type}.pdf', bbox_inches='tight')

In [ ]:
marker_map = {50: 'o', 100: 's', 200: '^'}

# make y-axis to bound/n, fraction
df_avg['Lower Bound of BP ratio'] = df_avg['Lower Bound of BP'] / df_avg['n']
df_avg['Upper Bound of BP ratio'] = df_avg['Upper Bound of BP'] / df_avg['n']

# 3. Single‐plot
plt.figure(figsize=(10, 6))
ax = sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Lower Bound of BP ratio',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend='full',
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Upper Bound of BP ratio',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend=False,
    alpha=0.8,
)

i = -1
for n_val in [50, 100, 200]:
    i += 1
    sub = df_avg[df_avg['n'] == n_val]
    ax.fill_between(sub['Effect Size'],
                    sub['Lower Bound of BP ratio'],
                    sub['Upper Bound of BP ratio'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'Effect Size $\theta$')
plt.ylabel('Breakdown Point of Rejection')
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False)
plt.tight_layout()
# plt.show()
plt.savefig(f'two_sample_bp_{loss_type}_ratio.pdf', bbox_inches='tight')

In [ ]:
# --- Main experiment: two-sample breakdown point analysis (template) ---
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from tqdm import tqdm

# Example parameters (adapt as needed for your two-sample setup)
loss_type = 'concordant'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.4811
ns = [50, 100, 200]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm]

# Example: two-sample effect sizes (adapt as needed)
effect_sizes = [-2, -1.5, -1, -.5, -.25, -.1, 0, .1, .25, .5, 1, 1.5, 2]

all_results = []
for effect in effect_sizes:
    for n in ns:
        for dist in dists:
            for seed in tqdm(seeds):
                np.random.seed(seed)
                phi0 = 0
                while phi0 == 0:
                    # --- Generate two samples ---
                    x1 = dist.rvs(size=n)
                    x2 = dist.rvs(size=n) + effect  # shift for effect size
                    phi0 = wald_test_two_sample(x1, x2, delta=delta, loss_type=loss_type)
                    if phi0 == 1:
                        bp_ub, _ = bound_power_upper_two_sample(x1, x2, delta=delta, loss_type=loss_type)
                        bp_lb, _ = bound_power_lower_two_sample(x1, x2, delta=delta, loss_type=loss_type)
                all_results.append({
                    'n': n,
                    'dist': getattr(dist, 'name', dist.__class__.__name__),
                    'seed': seed,
                    'bp_lb': bp_lb,
                    'bp_ub': bp_ub,
                    'effect': effect
                })

df = pd.DataFrame(all_results)

In [ ]:
# --- Aggregate and plot ---
df_avg = (
    df.groupby(['n', 'effect'], as_index=False)
    .agg({'bp_ub': 'mean', 'bp_lb': 'mean'})
)

df_avg = df_avg.rename(columns={
    'effect': 'Effect Size',
    'bp_ub': 'Upper Bound of BP',
    'bp_lb': 'Lower Bound of BP'
})

sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})  # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk") 
palette = sns.color_palette("husl", len(ns))
marker_map = {50: 'o', 100: 's', 200: '^'}

plt.figure(figsize=(10, 6))
ax = sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Lower Bound of BP',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend='full',
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Upper Bound of BP',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend=False,
    alpha=0.8,
)

i = -1
for n_val in [50, 100, 200]:
    i += 1
    sub = df_avg[df_avg['n'] == n_val]
    ax.fill_between(sub['Effect Size'],
                    sub['Lower Bound of BP'],
                    sub['Upper Bound of BP'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'Effect Size $\theta$')
plt.ylabel(r'$m$')
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False)
plt.tight_layout()
plt.savefig(f'two_sample_bp_{loss_type}.pdf', bbox_inches='tight')

In [ ]:
marker_map = {50: 'o', 100: 's', 200: '^'}

# make y-axis to bound/n, fraction
df_avg['Lower Bound of BP ratio'] = df_avg['Lower Bound of BP'] / df_avg['n']
df_avg['Upper Bound of BP ratio'] = df_avg['Upper Bound of BP'] / df_avg['n']

# 3. Single‐plot
plt.figure(figsize=(10, 6))
ax = sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Lower Bound of BP ratio',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend='full',
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Upper Bound of BP ratio',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend=False,
    alpha=0.8,
)

i = -1
for n_val in [50, 100, 200]:
    i += 1
    sub = df_avg[df_avg['n'] == n_val]
    ax.fill_between(sub['Effect Size'],
                    sub['Lower Bound of BP ratio'],
                    sub['Upper Bound of BP ratio'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'Effect Size $\theta$')
plt.ylabel('Breakdown Point of Rejection')
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False)
plt.tight_layout()
# plt.show()
plt.savefig(f'two_sample_bp_{loss_type}_ratio.pdf', bbox_inches='tight')